# Export the policy rules

Extracts the rules from the policy documents, builds one `Rule` per rule ID from `rules/rule_logic.yaml`, validates them, and exports them to `rules/ruleset.json`.

This uses no LLM, and nothing leaves your machine. Pick the `.venv` kernel, then run all cells.

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from expense_validator.compiler.build_rules import build_ruleset
from expense_validator.compiler.extract import extract_document
from expense_validator.compiler.validate import read_fields, validate_ruleset
from expense_validator.config import load_config
from expense_validator.policy.schema import RuleSet

paths = load_config().paths

## 1. Extract

In [2]:
documents = [extract_document(path.read_text(), path.name) for path in sorted(paths.policy_docs.glob("*.md"))]
clauses = [clause for document in documents for clause in document.clauses]

for document in documents:
    print(f"{document.policy_doc:<42}{len(document.clauses)} rules")
print(f"{'total':<42}{len(clauses)} rules")

approval-and-exception-handling.md        4 rules
gifts-and-awards-policy.md                4 rules
meals-and-entertainment-policy.md         7 rules
merchant-category-risk.md                 3 rules
non-reimbursable-items.md                 6 rules
receipt-documentation-requirements.md     4 rules
regional-addendum.md                      5 rules
technology-and-equipment-policy.md        4 rules
transaction-pattern-rules.md              4 rules
total                                     41 rules


## 2. Build

This stops with an error if a clause has no rule, a rule has no clause, a quote isn't in its clause, or a rule breaks the schema.

In [3]:
ruleset = build_ruleset(paths.rule_logic.read_text(), clauses)

print(f"version {ruleset.ruleset_version}: {len(ruleset.rules)} rules")
for kind, count in Counter(rule.kind for rule in ruleset.rules).items():
    print(f"  {kind}: {count}")

version v1: 42 rules
  process: 6
  check: 35
  carve_out: 1


## 3. Validate

Checks that rules read real fields, use their helpers properly, don't conflict, and match their clauses. Any error stops the export.

In [4]:
issues = validate_ruleset(ruleset, clauses, read_fields(paths.rule_fields.read_text()))
for issue in issues:
    print(issue)
errors = [issue for issue in issues if issue.level == "error"]
assert not errors, f"{len(errors)} errors: fix them before exporting"
print(f"{len(issues)} issues")

0 issues


## 4. Review

In [5]:
def outcome(rule):
    if rule.kind != "check":
        return rule.kind
    return rule.severity or "review signal"

def how(rule):
    if rule.status != "active":
        return rule.status
    return getattr(rule, "helper", None) or ("conditions" if rule.kind == "check" else "-")

def numbers(rule):
    found = [f"{param.value} {param.unit}" for param in rule.params.values() if param.type == "number"]
    return ", ".join(found) or "-"

print(f"{'RULE':<14}{'OUTCOME':<15}{'LOGIC':<24}NUMBERS")
for rule in sorted(ruleset.rules, key=lambda rule: rule.rule_id):
    print(f"{rule.rule_id:<14}{outcome(rule):<15}{how(rule):<24}{numbers(rule)}")

RULE          OUTCOME        LOGIC                   NUMBERS
ALC-01        high           conditions              2 count
ALC-02        medium         alcohol_per_attendee    30 usd
ALC-03        high           conditions              -
DOC-01        medium         conditions              75 usd
DOC-02        medium         conditions              -
DOC-03        medium         expense_type_mismatch   -
DOC-04        medium         conditions              -
ENT-01        medium         conditions              -
ENT-02        high           conditions              -
EXC-01        process        -                       -
EXC-02        process        -                       -
EXC-03        process        -                       0.5 ratio
EXC-04        process        -                       -
GIFT-01       medium         gift_cap                50 usd, 10 usd
GIFT-02       high           conditions              -
GIFT-03       medium         conditions              -
GIFT-04       critical

## 5. Export

In [6]:
paths.ruleset.write_text(ruleset.to_json())

reloaded = RuleSet.from_json(paths.ruleset.read_text())
assert reloaded.to_json() == ruleset.to_json()
print(f"Exported {len(reloaded.rules)} rules to {paths.ruleset.relative_to(ROOT)}")

Exported 42 rules to rules/ruleset.json


One exported rule, as it appears in the file:

In [7]:
exported = json.loads(paths.ruleset.read_text())
print(json.dumps(next(rule for rule in exported["rules"] if rule["rule_id"] == "TECH-02"), indent=2))

{
  "rule_id": "TECH-02",
  "policy_doc": "technology-and-equipment-policy.md",
  "title": "Peripherals under $200",
  "source_quote": "reimbursable up to **$200 per item**",
  "status": "active",
  "params": {
    "cap_usd": {
      "type": "number",
      "value": "200",
      "unit": "usd",
      "source": "policy"
    }
  },
  "provenance": {
    "translator": "developer",
    "clause_hash": "2e610562b99cd9a1f22e549ae0ed64b14058180245fc93a73c987d4e59574a98",
    "fingerprint": "ddb257d371040b974f664b12a9ac6683e1830ff89e6ef89e3ac1ba20f8198e58"
  },
  "kind": "check",
  "outcome": "exception",
  "scope": "item",
  "severity": "medium",
  "severity_cases": [],
  "select": {
    "op": "all",
    "conditions": [
      {
        "op": "eq",
        "field": "item.category",
        "value": "computer_accessory"
      },
      {
        "op": "gt",
        "field": "item.unit_price_usd",
        "value": {
          "param": "cap_usd"
        }
      }
    ]
  },
  "waived_by_ref": "pre_a